In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

random.seed(64)
np.random.seed(64)




In [2]:
def reduce_memory_usage(df):
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != "object":
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == "int":
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                else:
                    df[col] = df[col].astype(np.int64)
            else:
                if (
                    c_min > np.finfo(np.float16).min
                    and c_max < np.finfo(np.float16).max
                ):
                    df[col] = df[col].astype(np.float16)
                elif (
                    c_min > np.finfo(np.float32).min
                    and c_max < np.finfo(np.float32).max
                ):
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
        else:
            df[col] = df[col].astype("category")
    return df




In [3]:
train_path = "../input/tabular-playground-series-dec-2021/train.csv"
test_path = "../input/tabular-playground-series-dec-2021/test.csv"
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

train = reduce_memory_usage(train)
test = reduce_memory_usage(test)



In [4]:
print(f"dimensions of train: {train.shape}")
print(f"dimensions of test: {test.shape}")



dimensions of train: (3600000, 56)
dimensions of test: (400000, 55)


In [5]:
train["Cover_Type"] = train["Cover_Type"] - 1
x = train.drop(columns=["Id", "Cover_Type"])
y = train["Cover_Type"]



In [6]:
# Split and immediately cast to float32 numpy arrays for faster XGBoost ingestion
X_train_df, X_val_df, y_train, y_val = train_test_split(
    x, y, test_size=0.2, random_state=123, shuffle=True
)
X_train = X_train_df.to_numpy(dtype=np.float32, copy=False)
X_val = X_val_df.to_numpy(dtype=np.float32, copy=False)



In [7]:
test_df = test.drop(columns=["Id"])
X_test = test_df.to_numpy(dtype=np.float32, copy=False)



In [8]:
from xgboost import XGBClassifier

params = {
    "objective": "multi:softmax",
    "num_class": 7,
    "tree_method": "hist",  # CPU‑friendly histogram algorithm
    "eval_metric": "mlogloss",
    "booster": "gbtree",
    "gamma": 0.75,
    "max_depth": 7,
    "alpha": 10,
    "learning_rate": 0.007,
    "n_estimators": 500,  # fewer trees – still the same model type
    "predictor": "cpu_predictor",
    "n_jobs": -1,  # use all cores
}
xgb = XGBClassifier(**params)

xgb.fit(
    X_train,
    y_train,
    early_stopping_rounds=50,  # smaller patience – stops earlier if no gain
    eval_set=[(X_val, y_val)],
    verbose=False,
)



/usr/local/lib/python3.11/dist-packages/xgboost/sklearn.py:889: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [03:37:13] WARNING: /workspace/src/learner.cc:742: 
Parameters: { "predictor" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(alpha=10, base_score=None, booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, gamma=0.75, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.007, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=500,
              n_jobs=-1, num_class=7, ...)

In [9]:
y_val_pred = xgb.predict(X_val)
val_acc = accuracy_score(y_val, y_val_pred)
print("Validation Accuracy :", val_acc)



Validation Accuracy : 0.9427819444444444


In [10]:
test_pred = xgb.predict(X_test) + 1



In [11]:
submission_path = "../input/tabular-playground-series-dec-2021/sample_submission.csv"
submission = pd.read_csv(submission_path)
submission["Cover_Type"] = test_pred
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")
print(submission.head())

Submission saved to submission.csv
        Id  Cover_Type
0   814683           1
1  1357371           2
2  2106112           1
3  3483684           1
4  3960754           1
